# bench_med — Semi-Supervised Retinal OCT Classification

This notebook sets up a semi-supervised benchmark for retinal OCT disease
classification. We fine-tune the shared vision backbones from
`pipelines_torch.vision_models` using the `MeanTeacher` pseudo-labeling wrapper
in `pipelines_torch.ss_vision_models`.

- **Labeled source:** [Kermany2018](https://www.kaggle.com/datasets/paultimothymooney/kermany2018)
- **Unlabeled pool:** remaining images from Kermany's training split
- **Out-of-distribution test:** [Retinal OCT C8](https://www.kaggle.com/datasets/obulisainaren/retinal-oct-c8)

The pipeline preserves the existing dataset handling utilities while adding a
minimal semi-supervised training loop that reuses the shared components as much
as possible.


## 1. Imports and paths
We keep everything deterministic (`torch`, `numpy`, Python) and reuse the
registry models and SSL wrappers. The helper `RemappedImageFolder` filters the
OCT C8 dataset so we evaluate only on the class names shared with Kermany.


In [ ]:
import os
from pathlib import Path
from itertools import cycle
from typing import Dict

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm import tqdm

from sklearn.metrics import accuracy_score, f1_score

from pipelines_torch.vision_models import MODEL_REGISTRY
from pipelines_torch.ss_vision_models import MeanTeacher, PseudoLabel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
print(f"Using {DEVICE} for training")


In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
DATA_ROOT = Path("data_oct")
KERMANY_DIR = DATA_ROOT / "kermany2018"
OCTC8_DIR = DATA_ROOT / "oct_c8"

if not DATA_ROOT.exists():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)

try:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    if not KERMANY_DIR.exists():
        print("Downloading Kermany2018…")
        api.dataset_download_files("paultimothymooney/kermany2018", path=KERMANY_DIR, unzip=True)
    if not OCTC8_DIR.exists():
        print("Downloading Retinal OCT C8…")
        api.dataset_download_files("obulisainaren/retinal-oct-c8", path=OCTC8_DIR, unzip=True)
except Exception as exc:  # pragma: no cover - runtime environment dependent
    print(f"Kaggle download skipped or failed: {exc}")

train_root = KERMANY_DIR / "OCT2017" / "train"
val_root = KERMANY_DIR / "OCT2017" / "test"
if not train_root.exists() or not val_root.exists():
    raise FileNotFoundError("Expected OCT2017/train and OCT2017/test directories.")


In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.05)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = datasets.ImageFolder(train_root, transform=train_transform)
val_dataset = datasets.ImageFolder(val_root, transform=val_transform)
class_names = train_dataset.classes
NUM_CLASSES = len(class_names)
print(f"Train size: {len(train_dataset)}, Validation size: {len(val_dataset)}, Classes: {class_names}")


In [ ]:
# Split into labeled vs unlabeled pools
indices = np.random.RandomState(SEED).permutation(len(train_dataset))
lab_fraction = 0.2
lab_size = max(800, int(len(indices) * lab_fraction))
labeled_idx = indices[:lab_size]
unlabeled_idx = indices[lab_size:]

labeled_dataset = Subset(train_dataset, labeled_idx)
unlabeled_dataset = Subset(train_dataset, unlabeled_idx)

print(f"Using {len(labeled_dataset)} labeled images and {len(unlabeled_dataset)} unlabeled images")


In [ ]:
BATCH_L = 24
BATCH_U = 48
VAL_BATCH = 64

labeled_loader = DataLoader(labeled_dataset, batch_size=BATCH_L, shuffle=True, drop_last=True)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_U, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=VAL_BATCH, shuffle=False)


### Remapping helper for OCT C8
OCT C8 contains eight categories. We focus on the four classes shared with
Kermany2018 (`CNV`, `DME`, `DRUSEN`, `NORMAL`). The helper filters other classes
while preserving the label indices from the source dataset.


In [ ]:
class RemappedImageFolder(datasets.ImageFolder):
    def __init__(self, root: Path, transform, target_map: Dict[str, int]):
        super().__init__(root, transform=transform)
        filtered_samples = []
        for path, label in self.samples:
            cls_name = self.classes[label]
            if cls_name in target_map:
                filtered_samples.append((path, target_map[cls_name]))
        self.samples = filtered_samples
        self.targets = [t for _, t in filtered_samples]
        self.classes = list(target_map.keys())


train_class_map = {cls: idx for idx, cls in enumerate(class_names)}
octc8_map = {cls: train_class_map[cls] for cls in class_names}

octc8_eval_dir = OCTC8_DIR
if not octc8_eval_dir.exists():
    raise FileNotFoundError("Expected the OCT C8 dataset to be extracted under data_oct/oct_c8")

octc8_dataset = RemappedImageFolder(octc8_eval_dir, transform=val_transform, target_map=octc8_map)
octc8_loader = DataLoader(octc8_dataset, batch_size=VAL_BATCH, shuffle=False)
print(f"OCT C8 evaluation samples: {len(octc8_dataset)} (shared classes only)")


## 2. Semi-supervised training utilities
We wrap the chosen backbone (`timm_convnextv2_tiny` by default) with a
`MeanTeacher` SSL head. The training loop mirrors the structure of
`GeneralPipeline` while consuming labeled and unlabeled loaders.


In [ ]:
def create_ssl_model(backbone_name: str = "timm_convnextv2_tiny", *, use_mean_teacher: bool = True):
    if backbone_name not in MODEL_REGISTRY:
        raise KeyError(f"Unknown backbone {backbone_name}. Available: {list(MODEL_REGISTRY)}")
    backbone = MODEL_REGISTRY[backbone_name](num_classes=NUM_CLASSES)
    if use_mean_teacher:
        model = MeanTeacher(backbone, unsup_weight=1.0, rampup=5, ema_decay=0.996)
    else:
        model = PseudoLabel(backbone, threshold=0.95, unsup_weight=1.0, rampup=5)
    return model.to(DEVICE)


In [ ]:
def evaluate_model(model: torch.nn.Module, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_targets.append(yb.numpy())
    if not all_targets:
        return {"accuracy": float("nan"), "f1_macro": float("nan")}
    y_true = np.concatenate(all_targets)
    probs = np.concatenate(all_probs)
    y_pred = np.argmax(probs, axis=1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
    }


In [ ]:
def train_ssl(
    backbone_name: str = "timm_convnextv2_tiny",
    *,
    epochs: int = 10,
    lr: float = 3e-4,
    weight_decay: float = 1e-4,
    use_mean_teacher: bool = True,
):
    model = create_ssl_model(backbone_name, use_mean_teacher=use_mean_teacher)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = []
    unlabeled_iter = cycle(unlabeled_loader)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        n_steps = 0
        for xb_l, yb_l in labeled_loader:
            xb_u, _ = next(unlabeled_iter)
            xb_l = xb_l.to(DEVICE)
            yb_l = yb_l.to(DEVICE)
            xb_u = xb_u.to(DEVICE)

            optimizer.zero_grad()
            loss_sup, loss_unsup, logs = model.ssl_loss((xb_l, yb_l), (xb_u, None), epoch)
            loss = loss_sup + loss_unsup
            loss.backward()
            optimizer.step()
            if logs.get("_update_teacher") and hasattr(model, "update_teacher"):
                model.update_teacher()

            epoch_loss += loss.item()
            n_steps += 1

        metrics_val = evaluate_model(model, val_loader)
        metrics_oct = evaluate_model(model, octc8_loader)
        history.append({
            "epoch": epoch + 1,
            "loss": epoch_loss / max(1, n_steps),
            "val_accuracy": metrics_val["accuracy"],
            "val_f1_macro": metrics_val["f1_macro"],
            "oct_accuracy": metrics_oct["accuracy"],
            "oct_f1_macro": metrics_oct["f1_macro"],
        })
        print(
            f"Epoch {epoch+1:02d}: loss={history[-1]['loss']:.4f} "
            f"val_f1={history[-1]['val_f1_macro']:.3f} oct_f1={history[-1]['oct_f1_macro']:.3f}"
        )

    return model, pd.DataFrame(history)


## 3. Run the semi-supervised experiment
Set the `backbone_name` to any registry key (e.g. `timm_swinv2_small`) to compare
performance across architectures.


In [ ]:
BACKBONE = "timm_convnextv2_tiny"
EPOCHS = 8
LR = 3e-4

ssl_model, training_history = train_ssl(BACKBONE, epochs=EPOCHS, lr=LR, weight_decay=1e-4, use_mean_teacher=True)
training_history


### Evaluate on validation and OCT C8 splits
The final cell reuses the evaluation helper to double-check hold-out metrics.


In [ ]:
final_val = evaluate_model(ssl_model, val_loader)
final_oct = evaluate_model(ssl_model, octc8_loader)
print("Validation:", final_val)
print("OCT C8:", final_oct)


### Next steps
- Increase `epochs` or tweak the labeled ratio for deeper experiments.
- Swap `MeanTeacher` for `PseudoLabel` by setting `use_mean_teacher=False`.
- Add strong augmentations (e.g. RandAugment) by wrapping the dataset transforms.
